# Задачи с сайта leetcode.com


In [1]:
import sqlite3

import pandas as pd

con = sqlite3.connect("sqlite.db")
cur = con.cursor()

In [2]:
def execute_formatted_sqlite(sql_schema):
    sql = ";\n".join(sql_schema.split("\n"))
    cur.executescript(sql)
    con.commit()


def select(sql):
    return pd.read_sql(sql, con)

### 175. Combine Two Tables

Write a solution to report the first name, last name, city, and state of each person in the Person table. If the address of a personId is not present in the Address table, report null instead.

Return the result table in any order.


In [3]:
# SQLite3 не знает что такое TRUNCATE TABLE, поэтому нужно из запроса удалить эти команды.
# Добавить PRIMARY KEY к полям Id и OR REPLACE или IGNORE к INSERT, чтобы не дублировались строки при перезапуске.

sql_schema = """
Create table If Not Exists Person (personId int primary key, firstName varchar(255), lastName varchar(255))
Create table If Not Exists Address (addressId int primary key, personId int, city varchar(255), state varchar(255))
insert or replace into Person (personId, lastName, firstName) values ('1', 'Wang', 'Allen')
insert or replace into Person (personId, lastName, firstName) values ('2', 'Alice', 'Bob')
insert or replace into Address (addressId, personId, city, state) values ('1', '2', 'New York City', 'New York')
insert or replace into Address (addressId, personId, city, state) values ('2', '3', 'Leetcode', 'California')
"""

# А так же после кажной строки поставить ';'
sql = ";\n".join(sql_schema.split("\n"))
print(sql)

# Так как несколько команд, нужен executescript(), а не execute()
cur.executescript(sql)

# В SQLite3 в Python нет автокомита (в CLI есть), поэтому:
con.commit()

;
Create table If Not Exists Person (personId int primary key, firstName varchar(255), lastName varchar(255));
Create table If Not Exists Address (addressId int primary key, personId int, city varchar(255), state varchar(255));
insert or replace into Person (personId, lastName, firstName) values ('1', 'Wang', 'Allen');
insert or replace into Person (personId, lastName, firstName) values ('2', 'Alice', 'Bob');
insert or replace into Address (addressId, personId, city, state) values ('1', '2', 'New York City', 'New York');
insert or replace into Address (addressId, personId, city, state) values ('2', '3', 'Leetcode', 'California');



In [4]:
sql = """--sql
SELECT
    p.firstName,
    p.lastName,
    a.city,
    a.state
FROM
    person p
    LEFT JOIN Address a ON p.personid = a.personid
"""
select(sql)

,firstName,lastName,city,state
0,Allen,Wang,None,None
1,Bob,Alice,New York City,New York


### 176. Second Highest Salary

Write a solution to find the second highest distinct salary from the Employee table. If there is no second highest salary, return null (return None in Pandas).


In [5]:
sql_schema = """
Create table If Not Exists Employee (id int primary key, salary int)
insert or replace into Employee (id, salary) values ('1', '100')
insert or replace into Employee (id, salary) values ('2', '200')
insert or replace into Employee (id, salary) values ('3', '300')
"""
execute_formatted_sqlite(sql_schema)

In [6]:
# Если в таблице всего одна зарплата, OFFSET 1 не находит строк — запрос не возвращает вообще ничего.
# Вложив запрос в SELECT (...) — он вернёт строку, даже если внутри ничего не найдено.

sql = """--sql
SELECT
    (
        SELECT
            DISTINCT e.salary
        FROM
            employee e
        ORDER BY
            e.salary DESC
        LIMIT
            1
        OFFSET
            1
    ) AS second_highest_salary
"""
select(sql)

,second_highest_salary
0,200


### 177. Nth Highest Salary

Write a solution to find the nth highest distinct salary from the Employee table. If there are less than n distinct salaries, return null.


In [7]:
sql_schema = """
Create table If Not Exists Employee (Id int primary key, Salary int)
insert or replace into Employee (id, salary) values ('1', '100')
insert or replace into Employee (id, salary) values ('2', '200')
insert or replace into Employee (id, salary) values ('3', '300')
"""
execute_formatted_sqlite(sql_schema)

In [8]:
# Для PostgreSQL
# CREATE
# OR REPLACE FUNCTION NthHighestSalary (n INT) RETURNS INT AS $$ -- начало тела функции
# SELECT
#     CASE
#         WHEN n < 1 THEN NULL
#         ELSE (
#             SELECT
#                 salary
#             FROM
#                 (
#                     SELECT
#                         DISTINCT e.salary
#                     FROM
#                         employee e
#                     ORDER BY
#                         e.salary DESC
#                     LIMIT
#                         1
#                     OFFSET
#                         n -1
#                 ) AS result
#         )
#     END;

# $$ LANGUAGE SQL; -- конец тела функции с указанием используемого диалекта (чистый SQL)

# SELECT
#     NthHighestSalary(2);


# Имитация функции getNthHighestSalary для SQLite
def getNthHighestSalary(n):
    if n >= 1:
        sql = f"""--sql
        SELECT
            (
                SELECT DISTINCT
                    e.salary
                FROM
                    employee e
                ORDER BY
                    e.salary desc
                LIMIT
                    1
                OFFSET
                    {n - 1}
            ) AS "getNthHighestSalary({n})"
        """
    else:
        sql = f"""--sql
        SELECT
            NULL AS "getNthHighestSalary({n})"
        """
    return select(sql)


getNthHighestSalary(-1)

,getNthHighestSalary(-1)
0,None


### 178. Rank Scores

Write a solution to find the rank of the scores. The ranking should be calculated according to the following rules:

The scores should be ranked from the highest to the lowest.
If there is a tie between two scores, both should have the same ranking.
After a tie, the next ranking number should be the next consecutive integer value. In other words, there should be no holes between ranks.
Return the result table ordered by score in descending order.


In [9]:
sql_schema = """
Create table If Not Exists Scores (id int primary key, score DECIMAL(3,2))
insert or replace into Scores (id, score) values ('1', '3.5')
insert or replace into Scores (id, score) values ('2', '3.65')
insert or replace into Scores (id, score) values ('3', '4.0')
insert or replace into Scores (id, score) values ('4', '3.85')
insert or replace into Scores (id, score) values ('5', '4.0')
insert or replace into Scores (id, score) values ('6', '3.65')
"""
execute_formatted_sqlite(sql_schema)

In [10]:
sql = """--sql
SELECT
    s.score,
    DENSE_RANK() OVER (
        ORDER BY
            s.score desc
    ) AS rank
FROM
    scores s
"""
select(sql)

,score,rank
0,4.00,1
1,4.00,1
2,3.85,2
3,3.65,3
4,3.65,3
5,3.50,4


### 180. Consecutive Numbers

Find all numbers that appear at least three times consecutively.

Return the result table in any order.
